# Vehicle Claim Settlement Assistant

In [ ]:
"""
Date: 29 August 2024
Author: Swapnil Gaikwad
Aim: Build a insurance claim settlement system using GEN AI (LLM).
Background: Insurance industry is a very big insutry across the globe. Any insurance claim settlement process usually takes longer than expected
to come to the conclusion. There are many steps involved in this. 
This process can be made faster and can be automated to a great extent using Generative AI. 

Steps:

    Step 1: Get the data regarding an incident 
            1. user insurance policy
            2. accident report
            
    Step 2: Build a RAG system with traffic rules and regulations along with ACTS/Laws. This can be a very big pdf files/files. 
            As this information is very specific to country/city, and can get updated every year or so, it is better to build a RAG system
            
    Step 3: Create agents for:
            1. Getting summary from the text/reports
            2. finding the matching/similarity between reports
            3. Claim analyzer expert
            4. Claim settlement expert

    Step 4: Get summary of accident report
    Step 5: Query RAG with accident report summary, to get the laws related to the incident
    Step 6: Case Analysis - Get the analysis of the case based on accident report, customer insurance policy, laws involved
    Step 7: Take a judgement/conclusion from the LLM

    Step 8: In the end case analysis, judgement along with the data related to claim will be displayed on the web platform, 
    which will be further analyzed by the human insurance expert to make the final decision.
    
    
Tools used: 
    LLM - OpenAI
    Python stack - Llamma Index
    Vector DB - ChromaDB


Future Scope: As this is a prototype, there is a huge scope for improvements in the future.
            1. use agents system using CrewAI
            2. finetune open source LLM with the solved usecases, to make better decisions
            3. All the prompts can be improved
            4. This can be applied in all insurance sectors like health, Life, House, Travel insurance etc.
            5. Image to text - using LLM
"""


#### Install necessary libraries

In [ ]:
# Install dependencies for local Python 3.10 and Colab


### LLM

In [ ]:
import os
import openai
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "Set OPENAI_API_KEY in your environment before running this notebook. "
        "Do not paste secrets into notebook cells."
    )

openai.api_key = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")


In [ ]:
# OpenAI client is initialized in the previous cell.


In [ ]:
def chat(message, prompt):
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": f"{message}"}
        ]
    )
    text_only = response.choices[0].message.content
    return text_only


### Collect Claim related data

In [ ]:
import requests
import os


### Read vehicle acts and laws

In [ ]:
from pathlib import Path

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader


In [ ]:
vehicle_act_pdf = Path("India_motor_vehicle_act_1988.pdf")
if vehicle_act_pdf.exists():
    input_files = [str(vehicle_act_pdf)]
    print(f"Using vehicle act PDF: {vehicle_act_pdf.resolve()}")
else:
    sample_vehicle_act = Path("sample_vehicle_act.txt")
    sample_vehicle_act.write_text(
        "Sample motor vehicle rules for notebook execution. "
        "Drivers must obey traffic signals, speed limits, pedestrian crossings, "
        "and right-of-way rules. Dangerous or negligent driving that causes injury "
        "or property damage may affect claim liability and settlement decisions.\n"
    )
    input_files = [str(sample_vehicle_act)]
    print(
        "India_motor_vehicle_act_1988.pdf was not found. "
        f"Using fallback sample rules: {sample_vehicle_act.resolve()}"
    )

documents = SimpleDirectoryReader(input_files=input_files).load_data()
print(f"Loaded {len(documents)} document chunks.")


In [ ]:
len(documents)


In [ ]:
documents[:2]


### Accident report

In [ ]:
accident_report = """

Accident Report
Date: August 29, 2024
Time: 15:45 PM
Place: Intersection of Al Reem Street and Corniche Road, Abu Dhabi, UAE

People Involved:

Driver 1:

Name: Mr. Ahmed Al-Farsi
Age: 42
Gender: Male
Address: Al Reem Island, Abu Dhabi, UAE
Contact Number: +971-50-1234567
Driver's License Number: UAE-123456789
Vehicle: Toyota Land Cruiser 2021 (Black)
Registration Number: AD-1234
Driver 2:

Name: Mrs. Sara Al-Mansouri
Age: 35
Gender: Female
Address: Khalidiyah, Abu Dhabi, UAE
Contact Number: +971-50-7654321
Driver's License Number: UAE-987654321
Vehicle: Nissan Altima 2019 (White)
Registration Number: AD-5678
Pedestrian:

Name: Mr. Jamal Hassan
Age: 55
Gender: Male
Address: Tourist Club Area, Abu Dhabi, UAE
Contact Number: +971-55-1122334
Description of the Accident:

On August 29, 2024, at approximately 15:45 PM, a traffic accident occurred at the intersection of Al Reem Street and Corniche Road, Abu Dhabi. The incident involved two vehicles and one pedestrian.

Sequence of Events:

Driver 1 (Mr. Ahmed Al-Farsi) was traveling westbound on Al Reem Street in his Toyota Land Cruiser. According to the initial investigation, Mr. Al-Farsi was driving at a speed of approximately 70 km/h, exceeding the legal speed limit of 50 km/h in the area.
Driver 2 (Mrs. Sara Al-Mansouri) was driving her Nissan Altima and was approaching the same intersection from the opposite direction (eastbound) on Al Reem Street. Mrs. Al-Mansouri was in the process of making a left turn onto Corniche Road when the collision occurred.
Pedestrian (Mr. Jamal Hassan) was crossing Corniche Road at the pedestrian crosswalk at the time of the incident.
The Accident:

As Mr. Al-Farsi approached the intersection, the traffic signal was yellow, and he attempted to speed through the intersection before it turned red. Meanwhile, Mrs. Al-Mansouri had a green left-turn arrow and began making her turn.
Mr. Al-Farsi’s vehicle struck the right side of Mrs. Al-Mansouri’s vehicle at a speed of approximately 65 km/h. The force of the impact caused Mrs. Al-Mansouri’s vehicle to spin out of control and collide with a nearby traffic light pole.
During the collision, Mr. Jamal Hassan, the pedestrian, was also struck by Mrs. Al-Mansouri's vehicle as it veered off the road.
Injuries:

Mr. Ahmed Al-Farsi (Driver 1): Sustained minor injuries, including bruises on his chest and arms due to the deployment of the airbag. He was treated on-site by paramedics and later transported to Sheikh Khalifa Medical City for further examination.
Mrs. Sara Al-Mansouri (Driver 2): Sustained moderate injuries, including a fractured wrist and a concussion. She was conscious but disoriented at the scene and was transported to Cleveland Clinic Abu Dhabi for medical attention.
Mr. Jamal Hassan (Pedestrian): Sustained severe injuries, including multiple fractures to his legs and ribs, as well as internal bleeding. He was in critical condition and was rushed to the emergency room at Sheikh Khalifa Medical City.
Vehicle Damage:

Toyota Land Cruiser (Driver 1 - Mr. Ahmed Al-Farsi):

Front Bumper: Severely damaged
Hood: Dent and scratch marks
Right Headlight: Broken
Front Right Fender: Crumpled
Windshield: Cracked
Nissan Altima (Driver 2 - Mrs. Sara Al-Mansouri):

Right Side Door: Crumpled and torn off hinges
Right Side Mirror: Broken
Rear Bumper: Detached
Rear Right Wheel: Bent axle
Windshield: Shattered
Roof: Dent due to the impact with the traffic pole
Witness Statements:

Several witnesses confirmed that Mr. Al-Farsi was driving at a high speed and attempted to run the yellow light, leading to the collision. Other witnesses reported seeing Mrs. Al-Mansouri starting her turn when the signal was green for her direction.

Contributing Factors:

Speeding: Mr. Al-Farsi was exceeding the speed limit by 20 km/h.
Failure to Yield: Mr. Al-Farsi failed to yield to the oncoming traffic making a legal turn.
Signal Violation: Mr. Al-Farsi attempted to cross the intersection during a yellow signal.
Conclusion:

Based on the evidence collected, including witness statements, vehicle damages, and traffic signal data, it appears that Mr. Ahmed Al-Farsi was primarily at fault for the accident due to speeding and failing to yield the right of way. Further investigations are required to determine the full extent of liability.

Police Action:

Citation Issued to Mr. Ahmed Al-Farsi: For speeding, failure to yield, and traffic signal violation.
Investigation Continues: A detailed investigation will continue, and a final report will be submitted to the traffic court for further legal action.
"""


### Get customer insurance policy

In [ ]:
customer_vehicle_insurance_policy = """ 
Comprehensive Car Insurance Policy

Policyholder Information:

Name: Mrs. Sara Al-Mansouri
Address: Khalidiyah, Abu Dhabi, UAE
Contact Number: +971-50-7654321
Driver's License Number: UAE-987654321
Vehicle Information:

Make/Model: Nissan Altima
Year of Manufacture: 2019
Color: White
Engine CC/Fuel Type: 2488/Petrol
Registration Number: AD-5678
Insured Declared Value (IDV): AED 35,000
Policy Details:

Policy Number: DCCR10123867143/00
Policy Period: January 7, 2024, 05:30 AM to January 6, 2025, 11:59 PM
Policy Issuance Date: January 6, 2024, 01:13 PM
Previous Insurer: Acko General Insurance
No Claim Bonus (NCB): 45%
Coverage:

Own Damage Coverage:

Accident Coverage: Covers damages and losses to your vehicle resulting from accidents and collisions.
Fire Coverage: Covers damages and losses resulting from accidental fires.
Theft Coverage: Covers losses (up to the total declared insurance value) in the event of theft of your vehicle.
Calamities Coverage: Covers damages and losses resulting from natural calamities such as earthquakes, floods, and cyclones.
Third-Party Liability:

Third-Party Person Coverage: Covers financial or legal liabilities due to injuries or death caused to a third party.
Third-Party Property Coverage: Covers financial liabilities for damages caused to third-party property (up to AED 100,000).
Add-Ons:

Extra Car Protect:

Provides coverage for incidents such as key loss, outstation emergencies, and roadside assistance for car breakdowns.
Consumables Cover:

Covers the cost of consumables such as engine oils, screws, nuts, bolts, grease, and similar items during repairs.
Engine Protect:

Covers damages to the car's engine due to accidents, water ingression, or oil leakage.
NCB Protect:

Retains your current No Claim Bonus even in the event of a claim during the policy period.
Policy Exclusions:

Non-Accidental Damages: Damages due to wear and tear, breakdowns, and mechanical failures.
Tyres & Tubes: Regular wear and tear not covered, unless damaged in an accident (50% depreciation cut).
Undeclared Non-OEM Parts: Non-OEM parts like halogen bulbs, stereos, or bifuel kits not covered unless declared.
Commercial Use: Policy does not cover damages incurred while the vehicle is used for commercial purposes.
Claim Process:

In case of an accident, immediately notify Acko General Insurance. They will handle car repairs and deliver the repaired vehicle to your doorstep. Track the repair status in real-time via the Acko app.
"""


### RAG on vehicle insurance act and laws

#### VectorDB

In [ ]:
from llama_index.core import StorageContext
from llama_index.core.vector_stores.simple import SimpleVectorStore


In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

chroma_client = chromadb.PersistentClient(path="chroma")
collection_name = "IndianVehicleActLawsV1"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass
chroma_collection = chroma_client.create_collection(collection_name)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


In [ ]:
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)
query_engine = index.as_query_engine()


### Agents

#### summarization agent

#### Accident report summary

In [ ]:
prompt = "You are an expert at summarizing text in a concise and clear manner. It can be an email, or text extracted from pdf. It is related to insurance claim settlement in the insurance sector."
accident_report_summary = chat(accident_report, prompt)
print(accident_report_summary)


#### Get the matching traffic laws and acts

In [ ]:
#which taffic laws are involved
prompt="from the given accident report summary, please find which traffic rules and laws are involved in the incident in a single line"
traffic_laws_involved = chat(accident_report_summary, prompt)
print(traffic_laws_involved)


In [ ]:
matching_acts_laws = query_engine.query(traffic_laws_involved)
print(matching_acts_laws)


#### find if the person claiming has broken any laws

In [ ]:
prompt=f""" Identify which laws are broken. The focus is on identifying violations based on specific sections of vehicle acts and laws, particularly in the context of traffic accidents.

matching vehicle laws:
{matching_acts_laws}

accident_report_summary:
{accident_report_summary}

"""

text = """ You are an expert at finding the broken traffic laws from the given information."""
broken_rules = chat(text, prompt)
print("brokoen rules --- \n", broken_rules)


### Case summary

In [ ]:
prompt=f""" 
"Extract detailed numbered points from the accident report, customer vehicle policy, and matching laws. We are going to display this case summary on the web portal where an expert can review it.

     Accident Report:
    {accident_report_summary}

    Customer Vehicle Policy:
    {customer_vehicle_insurance_policy}

    Vehicle Acts and Laws:
    {matching_acts_laws}
"""

text = """ You are an expert at extracting and summarizing key points from the given reports, policies in insurance sector """


In [ ]:
case_summary = chat(text, prompt)
print("case_summary --- \n", case_summary)


### CASE ANALYSIS

In [ ]:
# Use the agent to analyze the case
prompt=f"""Analyze the following case and provide judgment:

    Accident Report:
    {accident_report}

    Customer Vehicle Policy:
    {customer_vehicle_insurance_policy}

    Case Summary:
    {case_summary}

    Provide the analysis in the following format:
    1. Should the claim be settled or not?
    2. Reasons/points behind it.
    3. If yes, how much amount should be settled.
    """

text = "You are an expert at solving vehicle insurance claims with over 10,000 cases solved successfully. You have an 25 years of experience in the insurance sector."
case_analysis = chat(text, prompt)
print("Case Analysis and Judgment:", case_analysis)


### Interface for Expert

In [ ]:
import gradio as gr


In [ ]:
# Function to simply return the case_analysis
def display_analysis():
    return case_analysis

# Define the Gradio interface to display the case_analysis
iface = gr.Interface(
    fn=display_analysis,
    inputs=None,  # No inputs needed
    outputs="text",
    title="Case Analysis and Judgment",
    description="Display the case analysis and judgment for the vehicle insurance claim."
)

if os.getenv("LAUNCH_GRADIO", "false").lower() == "true":
    iface.launch()
else:
    print("Set LAUNCH_GRADIO=true to launch the local Gradio interface.")


### Expert Dashboard/portal

In [ ]:
import gradio as gr

# Function to analyze the case
def analyze_case(accident_report, customer_vehicle_insurance_policy, case_summary):
    prompt = f"""Analyze the following case and provide judgment:

    Accident Report:
    {accident_report}

    Customer Vehicle Policy:
    {customer_vehicle_insurance_policy}

    Case Summary:
    {case_summary}

    Provide the analysis in the following format:
    1. Should the claim be settled or not?
    2. Reasons/points behind it.
    3. If yes, how much amount should be settled.
    """

    # Text to describe the agent's expertise
    text = "You are an expert at solving vehicle insurance claims with over 10,000 cases solved successfully. You have 25 years of experience in the insurance sector."

    # Simulate the chat function (replace this with your actual chat function call)
    case_analysis = chat(text, prompt)
    
    return case_analysis

# Function to handle the approval of the analysis
def approve_case(case_analysis):
    return f"Case Analysis Approved:\n\n{case_analysis}"

# Function to handle the editing of the analysis
def edit_case(accident_report, customer_vehicle_insurance_policy, case_summary):
    return analyze_case(accident_report, customer_vehicle_insurance_policy, case_summary)

# Define the Gradio interface
with gr.Blocks() as iface:
    gr.Markdown("# Vehicle Insurance Claim Analyzer")
    gr.Markdown("Analyze vehicle insurance claims and provide a judgment on whether the claim should be settled, the reasons behind it, and the settlement amount.")

    accident_report = gr.Textbox(lines=10, placeholder="Enter the accident report here...", label="Accident Report")
    customer_vehicle_insurance_policy = gr.Textbox(lines=10, placeholder="Enter the customer vehicle insurance policy here...", label="Customer Vehicle Policy")
    case_summary = gr.Textbox(lines=10, placeholder="Enter the case summary here...", label="Case Summary")
    
    with gr.Row():
        analyze_button = gr.Button("Analyze")
    
    case_analysis_output = gr.Textbox(lines=10, placeholder="Case analysis will appear here...", label="Case Analysis")

    with gr.Row():
        approve_button = gr.Button("Approve")
        edit_button = gr.Button("Edit")

    analyze_button.click(analyze_case, [accident_report, customer_vehicle_insurance_policy, case_summary], case_analysis_output)
    approve_button.click(approve_case, case_analysis_output, case_analysis_output)
    edit_button.click(edit_case, [accident_report, customer_vehicle_insurance_policy, case_summary], case_analysis_output)

if os.getenv("LAUNCH_GRADIO", "false").lower() == "true":
    iface.launch()
else:
    print("Set LAUNCH_GRADIO=true to launch the local Gradio interface.")
